Task 5: Fine-Tuning DistilBERT for POS Tagging & Chunking

Install Required Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Install required packages
!pip install transformers datasets seqeval evaluate accelerate nltk -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [2]:

!pip uninstall -y datasets
!pip install datasets==2.18.0 transformers seqeval

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


Task 1: Dataset Selection

In [3]:
import nltk
from nltk.corpus import conll2000
import pandas as pd
from datasets import Dataset, DatasetDict
from collections import Counter
nltk.download('conll2000', quiet=True)
# Parse CoNLL-2000 into a list of dicts with tokens + tag lists
def parse_conll2000(tagged_sents):
    records = []
    for sent in tagged_sents:
        tokens, pos_tags, chunk_tags = [], [], []
        for word, pos, chunk in sent:
            tokens.append(word)
            pos_tags.append(pos)
            chunk_tags.append(chunk)
        records.append({'tokens': tokens, 'pos_tags': pos_tags, 'chunk_tags': chunk_tags})
    return records

train_sents = conll2000.iob_sents('train.txt')  # 8936 sentences
test_sents  = conll2000.iob_sents('test.txt')   # 2012 sentences

train_records = parse_conll2000(train_sents)
test_records  = parse_conll2000(test_sents)
split_idx = int(len(train_records) * 0.9)
val_records = train_records[split_idx:]
train_records = train_records[:split_idx]

# Build a Hugging Face DatasetDict for compatibility with the rest of the pipeline
raw_dataset = DatasetDict({
    'train':      Dataset.from_list(train_records),
    'validation': Dataset.from_list(val_records),
    'test':       Dataset.from_list(test_records)
})
print('Dataset structure:')
print(raw_dataset)
print('\nSample entry (train[0]):')
print(raw_dataset['train'][0])

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 8042
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 894
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 2012
    })
})

Sample entry (train[0]):
{'tokens': ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], 'pos_tags': ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NN', 'NNS', 'IN', 'NNP', ',', 'JJ', 'IN', 'NN', 'NN', ',', 'VB', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NNP', 'CC', 'NNP', 'POS', 'JJ', 'NNS', '.'], 'chunk_tags': ['B-NP', 'B-PP', 'B

In [4]:
all_splits = ['train', 'validation', 'test']

pos_label_names = sorted(set(
    tag
    for split in all_splits
    for ex in raw_dataset[split]
    for tag in ex['pos_tags']
))

chunk_label_names = sorted(set(
    tag
    for split in all_splits
    for ex in raw_dataset[split]
    for tag in ex['chunk_tags']
))

print(f'POS Tag labels ({len(pos_label_names)}): {pos_label_names}')
print(f'\nChunk Tag labels ({len(chunk_label_names)}): {chunk_label_names}')

POS Tag labels (44): ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']

Chunk Tag labels (23): ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


Task 2: Data Preprocessing

In [5]:
def make_label2id(label_names):
    return {label: i for i, label in enumerate(label_names)}

pos_label2id_map   = make_label2id(pos_label_names)
chunk_label2id_map = make_label2id(chunk_label_names)

def tokenize_and_align_labels(examples, label_column, label2id_map):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = []
    for i, str_labels in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_list = []
        for word_id in word_ids:
            if word_id is None:
                label_list.append(-100)                              # special token
            elif word_id != previous_word_id:
                label_list.append(label2id_map[str_labels[word_id]])  # first subword
            else:
                label_list.append(-100)                              # continuation subword
            previous_word_id = word_id
        all_labels.append(label_list)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

print('Label alignment function defined.')

Label alignment function defined.


In [6]:
from transformers import AutoTokenizer
import logging
logging.set_verbosity_error = lambda *a, **k: None  # suppress transformers logs

MODEL_CHECKPOINT = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f'Tokenizer loaded: {MODEL_CHECKPOINT}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased


In [7]:
pos_tokenized = raw_dataset.map(
    lambda ex: tokenize_and_align_labels(ex, 'pos_tags', pos_label2id_map),
    batched=True
)

chunk_tokenized = raw_dataset.map(
    lambda ex: tokenize_and_align_labels(ex, 'chunk_tags', chunk_label2id_map),
    batched=True
)

print('Tokenized POS dataset features:', pos_tokenized['train'].features)
print('\nSample — input_ids length:', len(pos_tokenized['train'][0]['input_ids']))
print('Sample — labels:', pos_tokenized['train'][0]['labels'])

Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

Tokenized POS dataset features: {'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'pos_tags': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'chunk_tags': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'labels': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}

Sample — input_ids length: 43
Sample — labels: [-100, 18, 13, 10, 18, 38, 26, 36, 31, 33, 10, 14, 18, 13, 18, 21, 13, 19, 5, 14, 13, 18, 18, 5, 33, 31, 33, 10, 14, 18, 13, 19, 8, 19, 23, -100, 14, -100, -100, 21, -100, 6, -100]



Task 3: Model Setup

In [8]:
from transformers import AutoModelForTokenClassification

# id2label: integer → string label (used for inference display)
pos_id2label  = {i: l for i, l in enumerate(pos_label_names)}
pos_label2id  = pos_label2id_map

pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_names),
    id2label=pos_id2label,
    label2id=pos_label2id
)

print(f'POS Model: {len(pos_label_names)} labels')
print(f'Label mapping sample: {dict(list(pos_id2label.items())[:5])}')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


POS Model: 44 labels
Label mapping sample: {0: '#', 1: '$', 2: "''", 3: '(', 4: ')'}


In [9]:
chunk_id2label = {i: l for i, l in enumerate(chunk_label_names)}
chunk_label2id = chunk_label2id_map

chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_names),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

print(f'Chunk Model: {len(chunk_label_names)} labels')
print(f'Label mapping sample: {dict(list(chunk_id2label.items())[:5])}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Chunk Model: 23 labels
Label mapping sample: {0: 'B-ADJP', 1: 'B-ADVP', 2: 'B-CONJP', 3: 'B-INTJ', 4: 'B-LST'}


Task 4: Training

In [10]:

from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Data collator pads batches dynamically to the longest sequence in the batch
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Load seqeval metric — standard metric for sequence labeling tasks
seqeval = evaluate.load("seqeval")

print("Data collator and seqeval metric ready.")


Data collator and seqeval metric ready.


In [11]:
def make_compute_metrics(label_names):
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        # Argmax over last dim to get predicted label IDs
        predictions = np.argmax(logits, axis=-1)

        # Convert IDs → label strings, skip -100 (special/subword tokens)
        true_labels = [
            [label_names[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [label_names[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall":    results["overall_recall"],
            "f1":        results["overall_f1"],
            "accuracy":  results["overall_accuracy"],
        }
    return compute_metrics

print("compute_metrics factory defined.")

compute_metrics factory defined.


In [12]:

def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy='epoch',   # renamed from evaluation_strategy in newer transformers
        save_strategy='epoch',
        load_best_model_at_end=True,
        logging_steps=100,
        report_to='none'
    )

print('TrainingArguments factory defined.')

TrainingArguments factory defined.


In [ ]:
pos_trainer = Trainer(
    model=pos_model,
    args=get_training_args('./pos_model'),
    train_dataset=pos_tokenized['train'],
    eval_dataset=pos_tokenized['validation'],
    processing_class=tokenizer,   # 'tokenizer' arg removed in newer transformers
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_names)
)

print('Starting POS Tagging training...')
pos_trainer.train()
print('POS Tagging training complete!')


Starting POS Tagging training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.152963,0.123154,0.956683,0.957460,0.957072,0.969919
2,0.104052,0.091560,0.964885,0.964773,0.964829,0.975202


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
chunk_trainer = Trainer(
    model=chunk_model,
    args=get_training_args('./chunk_model'),
    train_dataset=chunk_tokenized['train'],
    eval_dataset=chunk_tokenized['validation'],
    processing_class=tokenizer,   # 'tokenizer' arg removed in newer transformers
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_names)
)

print('Starting Chunking training...')
chunk_trainer.train()
print('Chunking training complete!')


** Task 7: Comparison between POS Tagging and Chunking**

POS Tagging

POS (Part-of-Speech) tagging assigns grammatical labels to each word.

Examples: Noun (NN), Verb (VB), Adjective (JJ)
It is a word-level classification task.
Helps understand the grammatical structure of a sentence.

Example:
John → Noun
runs → Verb
Chunking
Chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP).
It is a phrase-level classification task.
Helps in identifying sentence structure beyond individual words.

Example:
"John works" → NP + VP

Key Differences

Aspect	POS Tagging	Chunking
Level	Word-level	Phrase-level
Complexity	Easier	More complex
Output	Grammar tags	Phrase groups
Use case	Syntax understanding	Structure understanding

Conclusion
POS tagging is simpler and focuses on individual words, whereas chunking is more advanced and focuses on grouping words into meaningful phrases.

Task 8: Report / Blog

Differences between POS Tagging and Chunking

Part-of-Speech (POS) Tagging and Chunking are both important tasks in Natural Language Processing, but they operate at different levels.

POS Tagging assigns grammatical labels (such as noun, verb, adjective) to each individual word in a sentence. It is a word-level classification task and helps in understanding the syntactic role of each word.

Chunking, also known as shallow parsing, groups words into meaningful phrases such as noun phrases (NP), verb phrases (VP), etc. It is a phrase-level task and provides a higher-level understanding of sentence structure.

Key Difference:

POS tagging focuses on individual words, while chunking focuses on grouping words into phrases.

Challenges Faced

Subword Tokenization Mismatch: BERT's WordPiece tokenizer splits words into subwords (e.g., "working" → ["work", "##ing"]). Since labels exist at the word level, careful alignment was needed — only the first subword receives the word's label; the rest get -100 to be ignored by the loss.

Special Token Handling: [CLS] and [SEP] tokens have no corresponding labels and must be masked with -100.

BIO Scheme Evaluation:
 seqeval evaluates entire spans, not individual tokens — a misclassified B tag breaks the entire phrase, making chunking evaluation stricter than POS.

Class Imbalance:
The O (Outside) tag dominates in chunking, which can bias the model toward over-predicting O.

Observations & Insights

DistilBERT achieved competitive results with only 40% fewer parameters than BERT-base — making it a practical choice for resource-constrained environments.
POS tagging converges faster and achieves higher F1 scores because each word has an independent label.

Chunking benefits significantly from BERT's contextual embeddings since phrase boundaries depend on multi-word context.

The DataCollatorForTokenClassification class was essential — it handles dynamic padding while correctly propagating -100 labels.

Pre-trained BERT representations capture rich syntactic knowledge, which is why even 3 epochs of fine-tuning yields strong token classification performance.


Conclusion

This assignment demonstrated that transformer models like BERT are highly powerful for sequence labeling tasks such as POS tagging and chunking. With proper preprocessing and training, high accuracy can be achieved in NLP applications.